In [1]:
from deepfix_sdk import DeepFixClient
import os

d:\workspace\repos\deepfix\.venv\Lib\site-packages\deepchecks\core\serialization\dataframe\html.py:16: UserWarning:

pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.



In [ ]:
os.environ["DEEPFIX_API_KEY"] = "sk-your-api-key"

In [2]:
client = DeepFixClient(api_url="http://deepfix.delcaux.com", timeout=120)

# Tabular data

## Classification

In [5]:
from deepfix_sdk.zoo.datasets import load_adult_classification
from deepfix_sdk.data.datasets import TabularDataset

In [6]:
train, test = load_adult_classification(as_train_test=True)
dataset_name = "adult-classification"
train_data = TabularDataset(dataset=train, dataset_name=dataset_name)
val_data = TabularDataset(dataset=test, dataset_name=dataset_name)

In [7]:
client.ingest_dataset(
    dataset_name=dataset_name,
    data_type="tabular",
    train_data=train_data,
    test_data=val_data,
    train_test_validation=val_data is not None,
    data_integrity=True,
    batch_size=8,
    overwrite=True,
)

In [8]:
# Diagnose dataset
result = client.diagnose_dataset(dataset_name=dataset_name)

Output()

✓ Analysis complete!

In [ ]:
# Visualize results
result.to_text()

╭──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                               DEEPFIX ANALYSIS RESULT                                                │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────────── Summary ───────────────────────────────────────────────────────╮
│ The analysis reveals significant data quality issues including high feature correlation between education            │
│ attributes, conflicting labels in 0.01% of samples, and special character problems in categorical columns. These     │
│ issues require immediate attention through feature selection, label cleaning, and data standardization.              │
│ Additionally, the target variable shows formatting inconsistencies that need resolution. Despite these data quality  │
│ concerns, the train-test validation results are excellent with minimal drift, indicating the current split           │
│ methodology produces representative datasets. The dataset requires comprehensive data cleaning before model training │
│ to ensure optimal performance.                                                                                       │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                      Summary Statistics                                      
 Metric                          Value                                                        
 Total Findings                  8                                                            
 Agents Involved                 DeepchecksArtifactsAnalyzer, CrossArtifactReasoningAgent     
 Severity Distribution           MEDIUM: 3  LOW: 3  HIGH: 2                                   

                                                HIGH Severity Issues (2)                                                
┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ # ┃ Agent                         ┃ Finding                                ┃ Action                                  ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1 │ DeepchecksArtifactsAnalyzer   │ High feature correlation between       │ Remove one of the highly correlated     │
│   │                               │ education and education-num            │ features (education or education-num)   │
│   │                               │ Evidence: Correlation greater than 0.9 │ to avoid multicollinearity and          │
│   │                               │ for pairs [('education',               │ redundant information                   │
│   │                               │ 'education-num')] detected in Feature  │ Highly correlated features provide      │
│   │                               │ Feature Correlation check              │ duplicate information which can confuse │
│   │                               │                                        │ the model, increase variance, and lead  │
│   │                               │                                        │ to unstable coefficient estimates       │
│   │                               │                                        │ without adding predictive value         │
│ 2 │ CrossArtifactReasoningAgent   │ High feature correlation and data      │ Remove redundant education feature,     │
│   │                               │ quality issues in categorical columns  │ clean conflicting labels, and           │
│   │                               │ Evidence: Education and education-num  │ standardize categorical values by       │
│   │                               │ show correlation >0.9, 0.01% samples   │ replacing special characters with       │
│   │                               │ have identical features but            │ appropriate missing value indicators    │
│   │                               │ conflicting labels, and categorical    │ High correlation causes                 │
│   │                               │ columns (workclass: 5.64%, occupation: │ multicollinearity, conflicting labels   │
│   │                               │ 5.66%, native-country: 1.79%) contain  │ create model ambiguity, and special     │
│   │                               │ special-character-only values          │ characters interfere with proper data   │
│   │                               │                                        │ processing and model training           │
└───┴───────────────────────────────┴────────────────────────────────────────┴─────────────────────────────────────────┘

                                               MEDIUM Severity Issues (3)                                               
┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ # ┃ Agent                         ┃ Finding                                ┃ Action                                  ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1 │ DeepchecksArtifactsAnalyzer   │ Conflicting labels present in dataset  │ Review and clean conflicting label      │
│   │                               │ Evidence: 0.01% of samples have        │ cases to ensure consistent labeling for │
│   │                               │ identical feature values but different │ identical feature combinations          │
│   │                               │ labels, indicating labeling            │ Conflicting labels create ambiguity in  │
│   │                               │ inconsistencies                        │ the training data, forcing the model to │
│   │                               │                                        │ learn contradictory patterns and        │
│   │                               │                                        │ reducing predictive accuracy            │
│ 2 │ DeepchecksArtifactsAnalyzer   │ Significant special characters in      │ Clean and standardize categorical       │
│   │                               │ categorical columns                    │ values by replacing                     │
│   │                               │ Evidence: workclass (5.64%),           │ special-character-only entries with     │
│   │                               │ occupation (5.66%), and native-country │ appropriate missing value indicators or │
│   │                               │ (1.79%) contain values with only       │ meaningful values                       │
│   │                               │ special characters                     │ Special character entries represent     │
│   │                               │                                        │ data quality issues that can interfere  │
│   │                               │                                        │ with proper categorical encoding and    │
│   │                               │                                        │ model interpretation                    │
│ 3 │ CrossArtifactReasoningAgent   │ Target variable formatting             │ Standardize income column values to     │
│   │                               │ inconsistency                          │ ensure consistent formatting (e.g.,     │
│   │                               │ Evidence: Income column contains       │ '<=50K' and '>50K')                     │
│   │                               │ variant '50k' which may not match      │ Inconsistent target variable formatting │
│   │                               │ expected formatting patterns           │ can cause processing errors and         │
│   │                               │                                        │ misclassification during model          │
│   │                               │                                        │ operations                              │
└───┴───────────────────────────────┴────────────────────────────────────────┴─────────────────────────────────────────┘

                                                LOW Severity Issues (3)                                                 
┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ # ┃ Agent                         ┃ Finding                                ┃ Action                                  ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1 │ DeepchecksArtifactsAnalyzer   │ String variant inconsistency in target │ Standardize the income column values to │
│   │                               │ column                                 │ ensure consistent formatting (e.g.,     │
│   │                               │ Evidence: Income column contains       │ ensure all values follow the same       │
│   │                               │ variant '50k' which may not match      │ pattern like '<=50K' and '>50K')        │
│   │                               │ expected formatting                    │ Inconsistent string formatting in the   │
│   │                               │                                        │ target variable can cause processing    │
│   │                               │                                        │ errors and misclassification during     │
│   │                               │                                        │ model training and prediction           │
│ 2 │ DeepchecksArtifactsAnalyzer   │ Excellent train-test validation with   │ Maintain current train-test split       │
│   │                               │ minimal drift                          │ methodology as it produces              │
│   │                               │ Evidence: All train-test validation    │ representative and well-balanced        │
│   │                               │ checks passed with minimal drift       │ datasets                                │
│   │                               │ scores (multivariate drift: 0.00421,   │ The consistent distribution between     │
│   │                               │ label drift: 0, feature drift scores < │ train and test sets ensures reliable    │
│   │                               │ 0.009)                                 │ performance evaluation and minimizes    │
│   │                               │                                        │ data leakage risks                      │
│ 3 │ CrossArtifactReasoningAgent   │ Excellent train-test validation with   │ Maintain current train-test split       │
│   │                               │ minimal data drift                     │ methodology for reliable model          │
│   │                               │ Evidence: All train-test validation    │ evaluation                              │
│   │                               │ checks passed with minimal drift       │ Consistent distribution between train   │
│   │                               │ scores (multivariate drift: 0.00421,   │ and test sets ensures accurate          │
│   │                               │ label drift: 0, feature drift scores < │ performance assessment and minimizes    │
│   │                               │ 0.009)                                 │ data leakage risks                      │
└───┴───────────────────────────────┴────────────────────────────────────────┴─────────────────────────────────────────┘

# Computer vision

## Image classification

In [10]:
from deepfix_sdk.zoo.datasets.foodwaste import load_train_and_val_datasets
from deepfix_sdk.data.datasets import ImageClassificationDataset

In [11]:
dataset_name = "cafetaria-foodwaste-lstroetmann"
# Load image datasets
train_data, val_data = load_train_and_val_datasets(
    image_size=448,
    batch_size=8,
    num_workers=4,
    pin_memory=False,
)
train_data = ImageClassificationDataset(dataset_name=dataset_name, dataset=train_data)
val_data = ImageClassificationDataset(dataset_name=dataset_name, dataset=val_data)

Getting label mapping: 100%|██████████| 375/375 [00:03<00:00, 99.82it/s] 


In [12]:
client.ingest_dataset(
    dataset_name=dataset_name,
    data_type="vision",
    train_data=train_data,
    test_data=val_data,
    train_test_validation=val_data is not None,
    data_integrity=True,
    batch_size=8,
    overwrite=True,
)

Computing dataset base statistics: 100%|██████████| 160/160 [00:08<00:00, 19.34it/s]


In [13]:
# Diagnose dataset
result = client.diagnose_dataset(dataset_name=dataset_name)

Output()

✓ Analysis complete!

In [ ]:
# Visualize results
result.to_text()

╭──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                               DEEPFIX ANALYSIS RESULT                                                │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────────── Summary ───────────────────────────────────────────────────────╮
│ The cross-artifact analysis reveals a fundamentally flawed dataset with catastrophic train-test split issues. The    │
│ Deepchecks analysis shows severe distribution mismatches across image properties (5/7 properties with high drift     │
│ scores) and a complete label distribution failure where 75% of test labels are unseen during training. Combined with │
│ the DatasetArtifactsAnalyzer's technical failure preventing comprehensive statistics review, and weak feature-target │
│ relationships, this dataset requires complete reconstruction before any meaningful machine learning can proceed. The │
│ primary recommendation is to recreate the dataset split using proper stratified sampling techniques to ensure        │
│ compatible distributions between training and testing data.                                                          │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                      Summary Statistics                                      
 Metric                          Value                                                        
 Total Findings                  7                                                            
 Agents Involved                 DeepchecksArtifactsAnalyzer, CrossArtifactReasoningAgent     
 Severity Distribution           HIGH: 4  MEDIUM: 3                                           

                                                HIGH Severity Issues (4)                                                
┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ # ┃ Agent                         ┃ Finding                                ┃ Action                                  ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1 │ DeepchecksArtifactsAnalyzer   │ Severe train-test distribution         │ Re-evaluate and recreate the train-test │
│   │                               │ mismatch                               │ split using proper randomization        │
│   │                               │ Evidence: Image Property Drift check   │ The extreme drift scores indicate the   │
│   │                               │ failed with 5/7 properties showing     │ current split creates fundamentally     │
│   │                               │ high drift scores (Brightness: 0.42,   │ different distributions, making         │
│   │                               │ RMS Contrast: 0.5, RGB intensities:    │ reliable model evaluation impossible    │
│   │                               │ 0.84-0.96), far exceeding the 0.2      │                                         │
│   │                               │ threshold                              │                                         │
│ 2 │ DeepchecksArtifactsAnalyzer   │ Catastrophic label distribution and    │ Ensure proper stratified sampling or    │
│   │                               │ new labels issue                       │ reorganize dataset to include all       │
│   │                               │ Evidence: 75% of test set labels were  │ classes in both sets                    │
│   │                               │ not in training set, with Cramer's V   │ Testing on completely unseen classes    │
│   │                               │ score of 0.92 indicating completely    │ makes model evaluation meaningless and  │
│   │                               │ different class distributions          │ prevents proper generalization          │
│   │                               │                                        │ assessment                              │
│ 3 │ CrossArtifactReasoningAgent   │ Catastrophic train-test data split     │ Completely recreate the dataset split   │
│   │                               │ failure                                │ using proper stratified sampling        │
│   │                               │ Evidence: DatasetArtifactsAnalyzer     │ techniques ensuring all classes are     │
│   │                               │ failed due to technical issues, but    │ represented in both training and test   │
│   │                               │ Deepchecks analysis reveals: 1) 5/7    │ sets                                    │
│   │                               │ image properties show severe drift     │ The current split creates fundamentally │
│   │                               │ scores (0.42-0.96) exceeding 0.2       │ incompatible distributions between      │
│   │                               │ threshold, 2) 75% of test labels not   │ training and testing, making any model  │
│   │                               │ present in training set with Cramer's  │ evaluation meaningless and preventing   │
│   │                               │ V score of 0.92 indicating completely  │ reliable generalization assessment      │
│   │                               │ different class distributions          │                                         │
│ 4 │ CrossArtifactReasoningAgent   │ Critical data integrity assessment gap │ Run comprehensive data integrity checks │
│   │                               │ Evidence: Data integrity section shows │ including outlier detection, label      │
│   │                               │ no results from Deepchecks analysis,   │ consistency validation, and complete    │
│   │  

                                               MEDIUM Severity Issues (3)                                               
┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ # ┃ Agent                         ┃ Finding                                ┃ Action                                  ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1 │ DeepchecksArtifactsAnalyzer   │ Missing data integrity validation      │ Run comprehensive data integrity checks │
│   │                               │ results                                │ including outlier detection and label   │
│   │                               │ Evidence: Data integrity section shows │ consistency validation                  │
│   │                               │ no results, suggesting potential       │ Without integrity checks, underlying    │
│   │                               │ issues with outlier detection or data  │ data quality issues may be hidden and   │
│   │                               │ quality checks                         │ contribute to model performance         │
│   │                               │                                        │ problems                                │
│ 2 │ DeepchecksArtifactsAnalyzer   │ Potential feature-target relationship  │ Analyze feature importance and consider │
│   │                               │ instability                            │ feature engineering to strengthen       │
│   │                               │ Evidence: Property Label Correlation   │ predictive signals                      │
│   │                               │ check passed with 0 PPS, suggesting    │ Weak feature-target relationships       │
│   │                               │ weak or inconsistent relationships     │ combined with data split issues         │
│   │                               │ between image properties and labels    │ compound model training difficulties    │
│ 3 │ CrossArtifactReasoningAgent   │ Weak feature-target relationships      │ After fixing data split, conduct        │
│   │                               │ compounding split issues               │ thorough feature analysis and           │
│   │                               │ Evidence: Property Label Correlation   │ engineering to identify and strengthen  │
│   │                               │ check passed with 0 PPS score,         │ predictive signals                      │
│   │                               │ indicating weak relationships between  │ Weak feature-target relationships       │
│   │                               │ image properties and labels,           │ combined with data split issues create  │
│   │                               │ exacerbating the impact of the poor    │ compounded challenges for model         │
│   │                               │ data split                             │ training and evaluation                 │
└───┴───────────────────────────────┴────────────────────────────────────────┴─────────────────────────────────────────┘

## Object detection

In [15]:
from deepfix_sdk.data.datasets import ObjectDetectionDataset

In [16]:
dataset_name = "general_dataset"
train_data = ObjectDetectionDataset.from_coco(
    dataset_name=dataset_name,
    images_directory_path=r"D:\workspace\general_dataset\coco\train",
    annotations_path=r"D:\workspace\general_dataset\coco\annotations\annotations_train.json",
)
val_data = ObjectDetectionDataset.from_coco(
    dataset_name=dataset_name,
    images_directory_path=r"D:\workspace\general_dataset\coco\val",
    annotations_path=r"D:\workspace\general_dataset\coco\annotations\annotations_val.json",
)

In [17]:
client.ingest_dataset(
    dataset_name=dataset_name,
    data_type="vision",
    train_data=train_data,
    test_data=val_data,
    train_test_validation=val_data is not None,
    data_integrity=True,
    batch_size=8,
    overwrite=True,
)

Computing base box statistics: 100%|██████████| 668/668 [00:00<00:00, 334462.82it/s]


In [18]:
# Diagnose dataset
result = client.diagnose_dataset(dataset_name=dataset_name)

Output()

✓ Analysis complete!

In [20]:
# Visualize results
result.to_text()

╭──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                               DEEPFIX ANALYSIS RESULT                                                │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────────── Summary ───────────────────────────────────────────────────────╮
│ Analysis reveals significant data quality issues despite technical limitations preventing full dataset statistics    │
│ extraction. The most critical finding is severe brightness distribution drift (KS score 0.29 vs 0.2 threshold)       │
│ indicating inconsistent lighting conditions between train and test datasets. Additional concerns include             │
│ inconsistent feature-target relationships for class sp6 and approaching class distribution drift. These issues       │
│ collectively undermine model validation reliability and require immediate attention to data collection consistency,  │
│ sampling methodology, and potential data augmentation before proceeding with model development.                      │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                      Summary Statistics                                      
 Metric                          Value                                                        
 Total Findings                  7                                                            
 Agents Involved                 DeepchecksArtifactsAnalyzer, CrossArtifactReasoningAgent     
 Severity Distribution           MEDIUM: 5  HIGH: 2                                           

                                                HIGH Severity Issues (2)                                                
┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ # ┃ Agent                         ┃ Finding                                ┃ Action                                  ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1 │ DeepchecksArtifactsAnalyzer   │ Significant brightness distribution    │ Investigate data collection process and │
│   │                               │ drift between train and test datasets  │ ensure consistent lighting conditions   │
│   │                               │ Evidence: Brightness property shows    │ across all data sources                 │
│   │                               │ Kolmogorov-Smirnov score of 0.29,      │ Different lighting conditions between   │
│   │                               │ exceeding the 0.2 threshold,           │ train and test sets create unreliable   │
│   │                               │ indicating different lighting          │ validation metrics and cause model      │
│   │                               │ conditions between datasets            │ performance degradation in production   │
│ 2 │ CrossArtifactReasoningAgent   │ Severe brightness distribution drift   │ Investigate and standardize data        │
│   │                               │ between train and test datasets        │ collection lighting conditions across   │
│   │                               │ indicating inconsistent lighting       │ all data sources and retrain model      │
│   │                               │ conditions                             │ Different lighting conditions between   │
│   │                               │ Evidence: Brightness property shows    │ datasets create unreliable validation   │
│   │                               │ Kolmogorov-Smirnov score of 0.29,      │ and cause significant model performance │
│   │                               │ exceeding the 0.2 threshold by 45%     │ degradation in production               │
└───┴───────────────────────────────┴────────────────────────────────────────┴─────────────────────────────────────────┘

                                               MEDIUM Severity Issues (5)                                               
┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ # ┃ Agent                         ┃ Finding                                ┃ Action                                  ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1 │ DeepchecksArtifactsAnalyzer   │ Changing feature-target relationship   │ Analyze class sp6 samples to identify   │
│   │                               │ for green intensity in class sp6       │ why green intensity correlation changed │
│   │                               │ Evidence: Predictive Power Score       │ and consider data augmentation to       │
│   │                               │ difference of 0.21 for Mean Green      │ normalize this relationship             │
│   │                               │ Relative Intensity property with class │ Inconsistent feature-target             │
│   │                               │ sp6, indicating inconsistent           │ relationships between datasets can lead │
│   │                               │ correlation patterns                   │ to models learning spurious             │
│   │                               │                                        │ correlations that don't generalize to   │
│   │                               │                                        │ new data                                │
│ 2 │ DeepchecksArtifactsAnalyzer   │ Approaching threshold class            │ Verify train-test split randomization   │
│   │                               │ distribution drift                     │ and consider stratified sampling to     │
│   │                               │ Evidence: Samples Per Class shows      │ maintain consistent class distributions │
│   │                               │ categorical drift score of 0.13, close │ Class distribution shifts can bias      │
│   │                               │ to the 0.15 failure threshold,         │ model performance metrics and make      │
│   │                               │ indicating potential class imbalance   │ validation results unreliable for       │
│   │                               │ issues                                 │ assessing true generalization           │
│ 3 │ CrossArtifactReasoningAgent   │ Dataset statistics unavailable due to  │ Fix the dataset statistics extraction   │
│   │                               │ technical error preventing             │ code and re-run the analysis to obtain  │
│   │                               │ comprehensive analysis                 │ complete dataset insights               │
│   │                               │ Evidence: DatasetArtifactsAnalyzer     │ Without proper dataset statistics,      │
│   │                               │ failed with AttributeError:            │ comprehensive data quality assessment   │
│   │                               │ 'DatasetArtifacts' object has no       │ cannot be completed, leaving potential  │
│   │                               │ attribute 'statistics'                 │ issues undetected                       │
│ 4 │ CrossArtifactReasoningAgent   │ Inconsistent feature-target            │ Analyze sp6 class samples to identify   │
│   │                               │ relationship for green intensity in    │ root cause and apply data               │
│   │                               │ class sp6 between datasets             │ normalization/augmentation techniques   │
│   │                               │ Evidence: Predictive Power Score       │ Inconsistent correlations between       │
│   │                               │ difference of 0.21 for Mean Green      │ datasets lead to models learning        │
│   │                               │ Relative Intensity property with class │ non-generalizable patterns that fail in │
│   │  

''

## Semantic segmentation

In [3]:
from deepfix_sdk.data.datasets import SemanticSegmentationDataset
from deepfix_sdk.zoo.datasets import load_segmentation_dataset

In [4]:
dataset_name = "coco_segmentation"
train_data, val_data = load_segmentation_dataset(
    batch_size=8,
    shuffle=False,
    pin_memory=False,
)
train_data = SemanticSegmentationDataset(dataset_name=dataset_name, dataset=train_data.dataset)
val_data = SemanticSegmentationDataset(dataset_name=dataset_name, dataset=val_data.dataset)


In [5]:
client.ingest_dataset(
    dataset_name=dataset_name,
    data_type="vision",
    train_data=train_data,
    test_data=val_data,
    train_test_validation=True,
    data_integrity=True,
    batch_size=8,
    overwrite=True,
)

Computing dataset base statistics: 100%|██████████| 49/49 [00:01<00:00, 36.15it/s]


In [8]:
# Diagnose dataset
result = client.diagnose_dataset(dataset_name=dataset_name)

Output()

✓ Analysis complete!

In [9]:
# Visualize results
result.to_text()

╭──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                               DEEPFIX ANALYSIS RESULT                                                │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────────── Summary ───────────────────────────────────────────────────────╮
│ The analysis reveals critical data quality issues with significant train-test dataset drift across multiple          │
│ dimensions. Key findings include image property drift (brightness, color intensity), label distribution              │
│ inconsistencies, and technical issues with dataset artifact structure. These problems collectively indicate that the │
│ current dataset splits are not properly randomized or representative, potentially leading to unreliable model        │
│ evaluation and overfitting. Recommendations focus on fixing data structure issues, implementing proper stratified    │
│ sampling methodologies, and establishing robust data validation pipelines to ensure dataset integrity and model      │
│ reliability.                                                                                                         │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                      Summary Statistics                                      
 Metric                          Value                                                        
 Total Findings                  6                                                            
 Agents Involved                 DeepchecksArtifactsAnalyzer, CrossArtifactReasoningAgent     
 Severity Distribution           HIGH: 3  MEDIUM: 3                                           

                                                HIGH Severity Issues (3)                                                
┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ # ┃ Agent                         ┃ Finding                                ┃ Action                                  ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1 │ DeepchecksArtifactsAnalyzer   │ Significant image property drift       │ Re-examine train-test split methodology │
│   │                               │ between train and test sets            │ to ensure proper randomization and      │
│   │                               │ Evidence: 3 out of 7 image properties  │ distribution matching                   │
│   │                               │ exceeded drift threshold: Brightness   │ The high drift scores indicate the test │
│   │                               │ (0.2), Mean Red Relative Intensity     │ set may not be representative of the    │
│   │                               │ (0.2), Mean Green Relative Intensity   │ training distribution, which can mask   │
│   │                               │ (0.24)                                 │ overfitting and lead to unreliable      │
│   │                               │                                        │ performance metrics                     │
│ 2 │ CrossArtifactReasoningAgent   │ Critical data integrity issues         │ Fix dataset artifact structure and      │
│   │                               │ detected in dataset splits with        │ implement comprehensive data validation │
│   │                               │ significant train-test drift           │ pipeline with proper train-test split   │
│   │                               │ Evidence: Deepchecks analysis revealed │ randomization                           │
│   │                               │ image property drift (brightness,      │ The combination of technical errors in  │
│   │                               │ red/green intensity exceeding          │ artifact analysis and confirmed data    │
│   │                               │ thresholds) and label distribution     │ drift indicates systemic data quality   │
│   │                               │ drift (Cramer's V score 0.17 > 0.15    │ issues that require immediate attention │
│   │                               │ threshold). DatasetArtifactsAnalyzer   │ to ensure model reliability             │
│   │                               │ failed due to missing statistics       │                                         │
│   │                               │ attribute, indicating potential data   │                                         │
│   │                               │ structure issues.                      │                                         │
│ 3 │ CrossArtifactReasoningAgent   │ Non-representative test set            │ Re-engineer train-test split using      │
│   │                               │ compromising model evaluation validity │ stratified sampling and validate        │
│   │                               │ Evidence: Multiple image properties    │ distribution matching across all        │
│   │                               │ showing significant drift (brightness  │ feature dimensions                      │
│   │                               │ 0.2, mean red intensity 0.2, mean      │ Current split methodology appears       │
│   │                               │ green intensity 0.24) suggests test    │ flawed, potentially masking overfitting │
│   │                               │ set may not accurately represent       │ and producing unreliable performance    │
│   │                               │ training distribution                  │ metrics that don't reflect real-world   │
│   │                               │                                        │ generalization                          │
└───┴──

                                               MEDIUM Severity Issues (3)                                               
┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ # ┃ Agent                         ┃ Finding                                ┃ Action                                  ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1 │ DeepchecksArtifactsAnalyzer   │ Label distribution drift with class    │ Implement stratified sampling or        │
│   │                               │ imbalance issues                       │ rebalance the dataset to ensure         │
│   │                               │ Evidence: Categorical drift detected   │ consistent class distributions across   │
│   │                               │ in 'Samples Per Class' with Cramer's V │ splits                                  │
│   │                               │ score of 0.17, exceeding the 0.15      │ Class distribution differences between  │
│   │                               │ threshold                              │ train and test sets can bias model      │
│   │                               │                                        │ evaluation and indicate potential       │
│   │                               │                                        │ sampling issues that affect             │
│   │                               │                                        │ generalization                          │
│ 2 │ DeepchecksArtifactsAnalyzer   │ Color property drift suggesting        │ Apply color augmentation techniques and │
│   │                               │ lighting/illumination differences      │ normalize color properties across the   │
│   │                               │ Evidence: Mean Red and Green Relative  │ dataset                                 │
│   │                               │ Intensity properties showing           │ Color property drift indicates the      │
│   │                               │ significant drift (0.2 and 0.24        │ model may be learning lighting-specific │
│   │                               │ respectively)                          │ features rather than generalizable      │
│   │                               │                                        │ patterns, increasing overfitting risk   │
│ 3 │ CrossArtifactReasoningAgent   │ Class imbalance and distribution       │ Implement class balancing techniques    │
│   │                               │ inconsistencies affecting model        │ and ensure consistent label             │
│   │                               │ fairness                               │ distributions across all dataset splits │
│   │                               │ Evidence: Categorical drift detected   │ Uneven class representation can bias    │
│   │                               │ in 'Samples Per Class' with Cramer's V │ model predictions and evaluation        │
│   │                               │ score of 0.17 exceeding the 0.15       │ metrics, particularly affecting         │
│   │                               │ threshold                              │ minority class performance              │
└───┴───────────────────────────────┴────────────────────────────────────────┴─────────────────────────────────────────┘

''

# NLP

## Sentence classification

In [10]:
from deepfix_sdk.zoo.datasets import load_tweet_emotion_classification
from deepfix_sdk.data.datasets import NLPDataset

In [11]:
train_data, test_data = load_tweet_emotion_classification(
        as_train_test=True, include_embeddings=True
    )
dataset_name = "tweet_emotion_classification"
train_data = NLPDataset(dataset_name=dataset_name, dataset=train_data)
test_data = NLPDataset(dataset_name=dataset_name, dataset=test_data)

In [12]:
client.ingest_dataset(
    dataset_name=dataset_name,
    data_type="nlp",
    train_data=train_data,
    test_data=test_data,
    train_test_validation=True,
    data_integrity=True,
    batch_size=8,
    overwrite=True,
)

deepchecks - WARNING - Could not find model's classes, using the observed classes. In order to make sure the classes used by the model are inferred correctly, please use the model_classes argument


In [13]:
# Diagnose dataset
result = client.diagnose_dataset(dataset_name=dataset_name)

Output()

✓ Analysis complete!

In [14]:
# Visualize results
result.to_text()

╭──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                               DEEPFIX ANALYSIS RESULT                                                │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────────── Summary ───────────────────────────────────────────────────────╮
│ The consolidated analysis reveals critical data quality issues that require immediate attention. The most severe     │
│ findings include significant label distribution drift between train and test sets (Cramer's V: 0.22) and a high      │
│ outlier ratio in toxicity data (16.43%), both indicating fundamental problems with data partitioning and quality.    │
│ Additional concerns include vocabulary mismatches (0.79% unknown tokens) and moderate embedding drift suggesting     │
│ domain shift. These issues collectively threaten model reliability and generalization, requiring comprehensive data  │
│ remediation before proceeding with model development or deployment.                                                  │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                      Summary Statistics                                      
 Metric                          Value                                                        
 Total Findings                  8                                                            
 Agents Involved                 DeepchecksArtifactsAnalyzer, CrossArtifactReasoningAgent     
 Severity Distribution           HIGH: 4  MEDIUM: 4                                           

                                                HIGH Severity Issues (4)                                                
┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ # ┃ Agent                         ┃ Finding                                ┃ Action                                  ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1 │ DeepchecksArtifactsAnalyzer   │ Significant label distribution drift   │ Re-evaluate train-test split            │
│   │                               │ between train and test sets            │ methodology and ensure both sets        │
│   │                               │ Evidence: Label drift score Cramer's V │ represent the same underlying           │
│   │                               │ of 0.22 exceeds the 0.15 threshold,    │ distribution                            │
│   │                               │ indicating substantial distribution    │ Label drift can lead to unreliable      │
│   │                               │ shift                                  │ performance metrics and poor            │
│   │                               │                                        │ generalization, as the model is trained │
│   │                               │                                        │ on a different distribution than it's   │
│   │                               │                                        │ evaluated on                            │
│ 2 │ DeepchecksArtifactsAnalyzer   │ High outlier ratio in toxicity         │ Investigate and clean toxicity          │
│   │                               │ property                               │ outliers, or implement robust outlier   │
│   │                               │ Evidence: Toxicity property shows      │ handling mechanisms                     │
│   │                               │ 16.43% outlier ratio, significantly    │ High outlier rates can distort feature  │
│   │                               │ exceeding the 5% threshold             │ distributions and model learning,       │
│   │                               │                                        │ particularly for sensitive properties   │
│   │                               │                                        │ like toxicity in emotion classification │
│ 3 │ CrossArtifactReasoningAgent   │ Severe label distribution drift        │ Re-evaluate and potentially recreate    │
│   │                               │ between train and test datasets        │ the train-test split using stratified   │
│   │                               │ Evidence: Label drift score Cramer's V │ sampling to ensure balanced label       │
│   │                               │ of 0.22 exceeds the 0.15 threshold,    │ distributions                           │
│   │                               │ indicating substantial distribution    │ Label drift fundamentally undermines    │
│   │                               │ shift that violates the assumption of  │ model evaluation validity and can lead  │
│   │                               │ identical train/test distributions     │ to false performance estimates and poor │
│   │                               │                                        │ generalization to real-world data       │
│ 4 │ CrossArtifactReasoningAgent   │ High outlier ratio in toxicity         │ Investigate toxicity outliers through   │
│   │                               │ property affecting data quality        │ exploratory analysis, implement         │
│   │                               │ Evidence: Toxicity property shows      │ appropriate outlier treatment (removal, │
│   │                               │ 16.43% outlier ratio, significantly    │ winsorization, or robust modeling       │
│   │                               │ exceeding the 5% acceptable threshold  │ techniques)                             │
│   │  

                                               MEDIUM Severity Issues (4)                                               
┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ # ┃ Agent                         ┃ Finding                                ┃ Action                                  ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1 │ DeepchecksArtifactsAnalyzer   │ Unknown tokens in text data            │ Update tokenizer vocabulary or          │
│   │                               │ Evidence: 0.79% unknown token ratio    │ preprocess text to handle               │
│   │                               │ indicates vocabulary mismatch between  │ out-of-vocabulary tokens                │
│   │                               │ data and tokenizer                     │ Unknown tokens can lead to information  │
│   │                               │                                        │ loss and reduced model performance in   │
│   │                               │                                        │ text classification tasks               │
│ 2 │ DeepchecksArtifactsAnalyzer   │ Moderate text embedding drift detected │ Monitor model performance closely and   │
│   │                               │ Evidence: Embedding drift value of 0.2 │ consider domain adaptation techniques   │
│   │                               │ (AUC 0.6) indicates some domain shift  │ if performance degrades                 │
│   │                               │ between train and test distributions   │ Embedding drift suggests the test data  │
│   │                               │                                        │ may come from a slightly different      │
│   │                               │                                        │ distribution, requiring careful         │
│   │                               │                                        │ monitoring and potential adaptation     │
│ 3 │ CrossArtifactReasoningAgent   │ Vocabulary mismatch with unknown       │ Update tokenizer vocabulary to include  │
│   │                               │ tokens in text data                    │ missing tokens or implement             │
│   │                               │ Evidence: 0.79% unknown token ratio    │ out-of-vocabulary handling strategies   │
│   │                               │ indicates significant vocabulary gaps  │ Unknown tokens result in information    │
│   │                               │ between the data and current tokenizer │ loss and reduced model performance,     │
│   │                               │                                        │ particularly critical for text          │
│   │                               │                                        │ classification tasks where vocabulary   │
│   │                               │                                        │ coverage is essential                   │
│ 4 │ CrossArtifactReasoningAgent   │ Moderate text embedding drift          │ Monitor model performance metrics       │
│   │                               │ indicating domain shift                │ closely and consider domain adaptation  │
│   │                               │ Evidence: Embedding drift value of 0.2 │ techniques if performance degradation   │
│   │                               │ (AUC 0.6) suggests some distributional │ is observed                             │
│   │                               │ differences between train and test     │ Embedding drift indicates potential     │
│   │                               │ data                                   │ domain shift that may require model     │
│   │                               │                                        │ adaptation or additional data           │
│   │                               │                                        │ collection to ensure robust performance │
└───┴──

''